# Context Embedding (Caption Video) — IndoBERTweet

Eksperimen tambahan: nambahin **caption video** sebagai konteks ke input model, selain komentar itu sendiri.

**Ide:** komentar "setuju banget dok" bisa berarti beda-beda tergantung video-nya ngomongin apa. Dengan nambahin caption video sebagai konteks, model (mudah-mudahan) bisa lebih ngerti maksud komentar tsb.

**Cara teknis:** pakai fitur bawaan BERT buat input sepasang kalimat — `tokenizer(caption, komentar)`. BERT otomatis nandain `token_type_ids` (segmen A = caption, segmen B = komentar), ini yang disebut *context embedding* lewat segment embedding bawaan arsitektur BERT.

**Keterbatasan penting (baca dulu sebelum lanjut):**
1. Baris **asli** (dari `dataset_final.csv`) punya `video_id`, jadi caption-nya bisa diambil dari `labeled_videos.csv`.
2. Baris **sintesis_paraphrase** / **dummy_baru** TIDAK punya `video_id` (dibuat dari teks komentar aja, bukan dari baris video tertentu). Jadi caption-nya dikosongin (`""`) untuk baris2 ini — model tetap bisa jalan, cuma context-nya "kosong" buat baris itu.
3. Caption biasanya panjang (ada deskripsi + hashtag), jadi `MAX_LEN` dinaikin dari 56 (punya yang tanpa caption) jadi **128**, dan truncation diset `only_first` supaya kalau kepanjangan, yang dipotong itu **caption duluan**, komentar (yang ada labelnya) tetap utuh.
4. Buat hemat waktu, notebook ini **skip 5-fold CV**, langsung training model final (full train.csv) lalu dievaluasi SEKALI ke `test.csv` — sama kaya alur `finetune_cv_final.py` pas `RUN_CV=False`.
5. Aturan anti-leakage tetap sama: validasi internal (buat early-stopping) cuma dari baris `asli`, sintesis/dummy selalu masuk training.

**WAJIB dijalanin di environment yang ada GPU + akses internet** (Colab paling gampang) — notebook ini download model dari HuggingFace Hub.


## 1. Setup & Konfigurasi

In [28]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, EarlyStoppingCallback,
    Trainer, TrainingArguments,
)

print('torch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())


torch: 2.14.1+cpu
CUDA tersedia: False


In [29]:
# ============================ KONFIGURASI ============================
SCHEMES = ['binary', '3class']
EPOCHS = 4
LR = 2e-5
BATCH_TRAIN = 16          # turunkan ke 8 kalau CUDA out of memory (caption bikin sequence lebih panjang)
BATCH_EVAL = 32
MAX_LEN = 128             # dinaikin dari 56 (versi tanpa caption) krn sekarang input = caption + komentar
WEIGHT_DECAY = 0.05       # sama kaya versi final finetune_cv_final.py (regularisasi lebih kuat)
WARMUP_RATIO = 0.1
EARLY_STOP_PATIENCE = 2
VAL_SIZE_INTERNAL = 0.1
SAVE_MODELS = False
SEED = 42
MODEL_NAME = 'indolem/indobertweet-base-uncased'
EARLY_STOP_METRIC = 'eval_loss'   # bukan f1_macro -- f1_macro bisa naik dikit2 walau val loss udah naik (overfit)
EARLY_STOP_GREATER_IS_BETTER = False

LABEL2ID = {
    'binary': {'non-negatif': 0, 'negatif': 1},
    '3class': {'negatif': 0, 'netral': 1, 'positif': 2},
}
LABEL_COL = {'binary': 'label_name', '3class': 'label_final'}
# =====================================================================


## 2. Cari Repo Root & Load Caption Video

In [30]:
def find_repo_root():
    current = Path.cwd()
    for _ in range(6):
        if (current / '01_pengumpulan_data').is_dir() and (current / '03_Indobert').is_dir():
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Nggak ketemu folder root repo (butuh '01_pengumpulan_data' dan '03_Indobert' "
        "keduanya ada sebagai subfolder). Jalankan dari dalam folder repo, atau pindahin "
        "notebook ini ke root repo."
    )
print('Repo root:', REPO_ROOT)

LABELED_VIDEOS_PATH = REPO_ROOT / '01_pengumpulan_data' / '05_scrap_komentar' / 'health_videos.csv'

Repo root: c:\Users\nesha\Tugas_akhir\seleksi-influencer


In [31]:
def load_captions(path):
    """Load health_videos.csv, ambil video_id + description (caption), dedup per video_id
    (1 video bisa ada >1 baris kalau di-scrape ulang -- ambil yg pertama ketemu)."""
    if not path.exists():
        raise FileNotFoundError(f'{path} tidak ditemukan.')
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8')
    required = {'video_id', 'description'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'health_videos.csv kurang kolom: {missing}')

    df = df[['video_id', 'description']].copy()
    df['video_id'] = df['video_id'].astype(str).str.strip()
    df['description'] = df['description'].fillna('').astype(str)
    df = df.drop_duplicates(subset='video_id', keep='first')
    df = df.rename(columns={'description': 'caption'})
    return df


captions_df = load_captions(LABELED_VIDEOS_PATH)
print(f'Caption ter-load: {len(captions_df)} video unik.')
captions_df.head()

Caption ter-load: 4428 video unik.


,video_id,caption
0,7677802473821588757,Nah ada lagi nih gebrakannya wardah! Cleanser ...
1,7667491760213544212,Tau gak kenapa dar*h bisa naik ke selang infus...
2,7662337718470397192,Jalan-jalan naik kudaMending simak pideo guee ...
3,7648588842672475410,"Pernah lagi nyaman2 mau tidur, eh tiba2 kerasa..."
4,7631539994821233940,yang mau tanya seputar imunisasi boleh komen y...


## 3. Load train/test + Gabungin Caption\n\nBaris `asli` dapet caption asli dari `video_id`-nya. Baris `sintesis_paraphrase`/`dummy_baru` ga punya `video_id` (kosong), jadi caption-nya ikut kosong -- ini dicek otomatis di bawah.

In [32]:
def load_train_test_with_caption(repo_root, scheme, captions_df):
    data_dir = repo_root / '03_Indobert' / 'Data' / scheme
    train_path = data_dir / 'train.csv'
    test_path = data_dir / 'test.csv'
    for p in (train_path, test_path):
        if not p.exists():
            raise FileNotFoundError(f'{p} tidak ditemukan. Jalankan gabung_final_training.py dulu.')

    train_df = pd.read_csv(train_path, dtype=str, keep_default_na=False, encoding='utf-8')
    test_df = pd.read_csv(test_path, dtype=str, keep_default_na=False, encoding='utf-8')

    label_col = LABEL_COL[scheme]
    label2id = LABEL2ID[scheme]

    for nama, df in [('train', train_df), ('test', test_df)]:
        required = {'text', 'sumber', label_col}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f'{scheme}/{nama}.csv kurang kolom: {missing}')
        if 'video_id' not in df.columns:
            raise ValueError(
                f"{scheme}/{nama}.csv ga ada kolom 'video_id' -- berarti file ini lahir dari "
                "versi gabung_final_training.py yang beda, cek lagi."
            )
        tidak_dikenal = set(df[label_col].unique()) - set(label2id.keys())
        if tidak_dikenal:
            raise ValueError(f'{scheme}/{nama}.csv punya label tak dikenal: {tidak_dikenal}')
        df['label_id'] = df[label_col].map(label2id)

    if 'fold' not in train_df.columns:
        train_df['fold'] = ''

    # --- Gabungin caption lewat video_id (left join -- baris tanpa video_id dapet caption kosong) ---
    for nama, df in [('train', train_df), ('test', test_df)]:
        df['video_id'] = df['video_id'].astype(str).str.strip()

    train_df = train_df.merge(captions_df, on='video_id', how='left')
    train_df['caption'] = train_df['caption'].fillna('')
    test_df = test_df.merge(captions_df, on='video_id', how='left')
    test_df['caption'] = test_df['caption'].fillna('')

    return train_df, test_df


def cek_cakupan_caption(df, nama):
    asli = df[df['sumber'] == 'asli']
    non_asli = df[df['sumber'] != 'asli']
    asli_ada_caption = (asli['caption'].str.strip() != '').mean() if len(asli) else 0
    non_asli_semua_kosong = (non_asli['caption'].str.strip() == '').all() if len(non_asli) else True
    print(f'[{nama}] baris asli dgn caption terisi: {asli_ada_caption * 100:.1f}% '
          f'({len(asli)} baris asli total)')
    print(f'[{nama}] baris sintesis/dummy semua caption-nya kosong (seharusnya True): {non_asli_semua_kosong}')
    if asli_ada_caption < 0.9:
        print(f'  PERHATIAN: banyak baris asli yg video_id-nya ga ketemu di labeled_videos.csv, cek lagi datanya.')


## 4. Fungsi Anti-Leakage (sama kayak `finetune_cv_final.py`)\n\nValidasi internal (early-stopping) cuma dari baris `asli` -- baris sintesis/dummy selalu masuk training, ga pernah ke validasi/held-out.

In [33]:
def compute_class_weights(label_ids, n_classes):
    label_ids = np.asarray(label_ids)
    counts = np.array([(label_ids == c).sum() for c in range(n_classes)])
    if (counts == 0).any():
        raise ValueError(f'Ada kelas dengan 0 anggota: counts={counts.tolist()}')
    n_samples = len(label_ids)
    weights = n_samples / (n_classes * counts)
    return weights


def split_asli_only_validation(df, label_col='label_id', val_size=VAL_SIZE_INTERNAL, seed=42):
    asli_mask = df['sumber'] == 'asli'
    asli_df = df[asli_mask]
    non_asli_df = df[~asli_mask]

    if len(asli_df) == 0:
        train_final_df, val_final_df = train_test_split(
            df, test_size=val_size, random_state=seed, stratify=df[label_col],
        )
        return train_final_df.reset_index(drop=True), val_final_df.reset_index(drop=True)

    try:
        asli_train_df, asli_val_df = train_test_split(
            asli_df, test_size=val_size, random_state=seed, stratify=asli_df[label_col],
        )
    except ValueError:
        asli_train_df, asli_val_df = train_test_split(
            asli_df, test_size=val_size, random_state=seed,
        )
    train_final_df = pd.concat([asli_train_df, non_asli_df], ignore_index=True)
    val_final_df = asli_val_df.reset_index(drop=True)
    return train_final_df, val_final_df


## 5. Dataset & Model (versi caption + komentar sebagai sepasang kalimat)

In [34]:
class CommentCaptionDataset(Dataset):
    """Tokenize SEPASANG teks: (caption, komentar). truncation=True pakai strategi
    longest_first -- motong dari sequence yg lagi lebih panjang. Karena caption hampir
    selalu jauh lebih panjang dari komentar, praktiknya caption yg kepotong duluan; komentar
    cuma ikut kepotong kalau captionnya kosong DAN komentarnya sendiri kepanjangan."""
    def __init__(self, captions, texts, labels, tokenizer, max_len):
        enc = tokenizer(
            list(captions), list(texts),
            truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt',
        )
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.token_type_ids = enc.get('token_type_ids', None)
        self.labels = torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx],
        }
        if self.token_type_ids is not None:
            item['token_type_ids'] = self.token_type_ids[idx]
        return item

In [35]:
def train_one_model(train_sub_df, val_df, label_names, class_weight_vec, tokenizer,
                     model_name, max_len, out_dir, seed_offset=0):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    n_labels = len(label_names)

    train_ds = CommentCaptionDataset(train_sub_df['caption'], train_sub_df['text'],
                                      train_sub_df['label_id'], tokenizer, max_len)
    val_ds = CommentCaptionDataset(val_df['caption'], val_df['text'],
                                    val_df['label_id'], tokenizer, max_len)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_labels)

    total_steps = (len(train_sub_df) // BATCH_TRAIN + 1) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    args = TrainingArguments(
        output_dir=str(out_dir / 'checkpoints'),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model=EARLY_STOP_METRIC,
        greater_is_better=EARLY_STOP_GREATER_IS_BETTER,
        fp16=(device == 'cuda'),
        logging_steps=50,
        seed=SEED + seed_offset,
        report_to='none',
        disable_tqdm=False,
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=make_compute_metrics(label_names),
        class_weights=class_weight_vec,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
    )
    trainer.train()
    return trainer


def evaluate_on(trainer, eval_df, label_names, tokenizer, max_len):
    ds = CommentCaptionDataset(eval_df['caption'], eval_df['text'], eval_df['label_id'], tokenizer, max_len)
    pred_out = trainer.predict(ds)
    preds = np.argmax(pred_out.predictions, axis=-1)
    labels_true = eval_df['label_id'].astype(int).to_numpy()
    report = classification_report(labels_true, preds, target_names=label_names,
                                    output_dict=True, zero_division=0)
    cm = confusion_matrix(labels_true, preds, labels=list(range(len(label_names))))
    return report, cm.tolist()


## 6. Training (langsung model final, skip CV)

In [36]:
MODELS_DIR = REPO_ROOT / '03_Indobert' / 'models_context'
RESULTS_DIR = REPO_ROOT / '03_Indobert' / 'Data' / 'hasil_context_embedding'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))
else:
    print('PERHATIAN: tidak terdeteksi GPU. Training akan sangat lambat di CPU.')

final_results = {}

for scheme in SCHEMES:
    label2id = LABEL2ID[scheme]
    label_names = [name for name, _ in sorted(label2id.items(), key=lambda kv: kv[1])]
    n_labels = len(label_names)

    train_df, test_df = load_train_test_with_caption(REPO_ROOT, scheme, captions_df)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print(f'\n{"=" * 70}\n  SKEMA: {scheme}  ({n_labels} kelas)  [+ context embedding caption]\n{"=" * 70}')
    print(f'  train total: {len(train_df)}  (asli={len(train_df[train_df.sumber == "asli"])}, '
          f'sintesis/dummy={len(train_df[train_df.sumber != "asli"])})')
    print(f'  test total : {len(test_df)}  (semua asli, hold-out sungguhan)')
    cek_cakupan_caption(train_df, f'{scheme}/train')
    cek_cakupan_caption(test_df, f'{scheme}/test')

    class_weight_vec_np = compute_class_weights(train_df['label_id'], n_labels)
    class_weight_vec = torch.tensor(class_weight_vec_np, dtype=torch.float)

    train_sub_df, val_internal_df = split_asli_only_validation(
        train_df, label_col='label_id', val_size=VAL_SIZE_INTERNAL, seed=SEED,
    )

    out_dir_final = MODELS_DIR / scheme / 'final'
    t0 = time.time()
    trainer_final = train_one_model(
        train_sub_df, val_internal_df,
        label_names, class_weight_vec, tokenizer, MODEL_NAME, MAX_LEN, out_dir_final,
    )
    print(f'  waktu training: {(time.time() - t0) / 60:.1f} menit')

    report_test, cm_test = evaluate_on(trainer_final, test_df, label_names, tokenizer, MAX_LEN)
    print(f'  [FINAL+caption -> test.csv] accuracy={report_test["accuracy"]:.3f} '
          f'f1_macro={report_test["macro avg"]["f1-score"]:.3f}')
    for name in label_names:
        print(f'    f1_{name} = {report_test[name]["f1-score"]:.3f}')

    report_train, cm_train = evaluate_on(trainer_final, train_df, label_names, tokenizer, MAX_LEN)
    print(f'  [FINAL+caption -> train.csv] accuracy={report_train["accuracy"]:.3f} '
          f'f1_macro={report_train["macro avg"]["f1-score"]:.3f}')

    final_results[scheme] = {
        'label_names': label_names,
        'train': {'report': report_train, 'confusion_matrix': cm_train},
        'test': {'report': report_test, 'confusion_matrix': cm_test},
    }

    if SAVE_MODELS:
        out_dir_final.mkdir(parents=True, exist_ok=True)
        trainer_final.save_model(str(out_dir_final))
        tokenizer.save_pretrained(str(out_dir_final))

    del trainer_final, tokenizer
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

with open(RESULTS_DIR / 'final_train_vs_test_context.json', 'w', encoding='utf-8') as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False)

print('\nTersimpan di 03_Indobert/Data/hasil_context_embedding/final_train_vs_test_context.json')


Device: cpu
PERHATIAN: tidak terdeteksi GPU. Training akan sangat lambat di CPU.

  SKEMA: binary  (2 kelas)  [+ context embedding caption]
  train total: 2218  (asli=1668, sintesis/dummy=550)
  test total : 417  (semua asli, hold-out sungguhan)
[binary/train] baris asli dgn caption terisi: 98.4% (1668 baris asli total)
[binary/train] baris sintesis/dummy semua caption-nya kosong (seharusnya True): True
[binary/test] baris asli dgn caption terisi: 98.3% (417 baris asli total)
[binary/test] baris sintesis/dummy semua caption-nya kosong (seharusnya True): True


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14278.54it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing 

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Non-negatif,F1 Negatif
1,0.308299,0.632099,0.916168,0.478125,0.956250,0.000000
2,0.191004,0.497189,0.928144,0.606132,0.962264,0.250000
3,0.114868,0.637104,0.922156,0.597142,0.958991,0.235294
4,0.105301,0.631996,0.922156,0.597142,0.958991,0.235294


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.89it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.95it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.78it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1

  waktu training: 50.2 menit


c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


... (5 outputs left out)

Could not reach the Hub ([Errno 11001] getaddrinfo failed). Using cached commit hash for 'indolem/indobertweet-base-uncased'.
[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16808.98it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias  

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Negatif,F1 Netral,F1 Positif
1,0.451971,0.735258,0.772455,0.511350,0.230769,0.870849,0.432432


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.68it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

## 7. Laporan Perbandingan Train vs Test (per skema, dgn context embedding)

In [ ]:
def tabel_classification_report(report, label_names):
    baris = []
    for name in label_names:
        r = report[name]
        baris.append({'kelas': name, 'precision': r['precision'], 'recall': r['recall'],
                      'f1-score': r['f1-score'], 'support': int(r['support'])})
    for key in ['macro avg', 'weighted avg']:
        r = report[key]
        baris.append({'kelas': key, 'precision': r['precision'], 'recall': r['recall'],
                      'f1-score': r['f1-score'], 'support': int(r['support'])})
    baris.append({'kelas': 'accuracy', 'precision': '', 'recall': '', 'f1-score': report['accuracy'],
                  'support': int(report['macro avg']['support'])})
    return pd.DataFrame(baris).set_index('kelas')


def tabel_perbandingan_train_test(report_train, report_test, label_names):
    baris = []
    for name in label_names + ['macro avg', 'weighted avg']:
        rt, rv = report_train[name], report_test[name]
        baris.append({
            'kelas': name,
            'precision_train': rt['precision'], 'precision_test': rv['precision'],
            'recall_train': rt['recall'], 'recall_test': rv['recall'],
            'f1_train': rt['f1-score'], 'f1_test': rv['f1-score'],
            'gap_f1 (test - train)': rv['f1-score'] - rt['f1-score'],
        })
    baris.append({
        'kelas': 'accuracy', 'precision_train': '', 'precision_test': '',
        'recall_train': '', 'recall_test': '',
        'f1_train': report_train['accuracy'], 'f1_test': report_test['accuracy'],
        'gap_f1 (test - train)': report_test['accuracy'] - report_train['accuracy'],
    })
    return pd.DataFrame(baris).set_index('kelas')


def tabel_confusion_matrix(cm, label_names):
    idx = [f'asli: {n}' for n in label_names]
    cols = [f'prediksi: {n}' for n in label_names]
    return pd.DataFrame(cm, index=idx, columns=cols)


def cetak_laporan_skema(scheme, data, float_fmt='{:.3f}'):
    label_names = data['label_names']
    report_train, report_test = data['train']['report'], data['test']['report']
    cm_train, cm_test = data['train']['confusion_matrix'], data['test']['confusion_matrix']

    pd.set_option('display.float_format', float_fmt.format)
    print('\n' + '=' * 70)
    print(f'SKEMA: {scheme.upper()}  (+ context embedding caption)')
    print('=' * 70)
    print('\n-- Classification report TRAIN --')
    print(tabel_classification_report(report_train, label_names).to_string())
    print('\n-- Classification report TEST --')
    print(tabel_classification_report(report_test, label_names).to_string())
    print('\n-- Perbandingan TRAIN vs TEST --')
    print(tabel_perbandingan_train_test(report_train, report_test, label_names).to_string())
    print('\n-- Confusion matrix TRAIN --')
    print(tabel_confusion_matrix(cm_train, label_names).to_string())
    print('\n-- Confusion matrix TEST --')
    print(tabel_confusion_matrix(cm_test, label_names).to_string())

    gap_acc = report_test['accuracy'] - report_train['accuracy']
    gap_f1 = report_test['macro avg']['f1-score'] - report_train['macro avg']['f1-score']
    print(f'\nGap accuracy (test - train): {gap_acc:+.3f}')
    print(f'Gap F1-macro (test - train): {gap_f1:+.3f}')
    if gap_f1 < -0.1:
        print('  -> PERHATIAN: gap > 0.1, kemungkinan model overfit ke training set.')
    else:
        print('  -> gap wajar, model keliatan generalize dengan oke.')


for scheme, data in final_results.items():
    cetak_laporan_skema(scheme, data)


## 8. Bandingin vs Model TANPA Caption (baseline)

Angka baseline (tanpa context embedding, dari `finetune_cv_final.py` yang udah dijalanin sebelumnya) di-isi manual di bawah -- **GANTI angka ini sesuai hasil terakhir kamu** kalau beda, lalu jalanin cell ini buat liat context embedding nolong apa kagak.

In [ ]:
# GANTI sesuai hasil final_train_vs_test.json (tanpa caption) yang kamu punya
baseline_tanpa_caption = {
    'binary': {
        'accuracy': 0.923,
        'f1_macro': 0.721,
        'f1_non-negatif': 0.959,
        'f1_negatif': 0.484,
    },
    # kalau punya angka 3class tanpa-caption juga, isi di sini
}

print('Perbandingan TANPA caption vs DENGAN context embedding (caption):\n')
for scheme, baseline in baseline_tanpa_caption.items():
    if scheme not in final_results:
        continue
    report_test = final_results[scheme]['test']['report']
    label_names = final_results[scheme]['label_names']
    print(f'--- {scheme.upper()} ---')
    print(f'  accuracy   : tanpa_caption={baseline["accuracy"]:.3f}  ->  dgn_caption={report_test["accuracy"]:.3f}  '
          f'(delta {report_test["accuracy"] - baseline["accuracy"]:+.3f})')
    print(f'  f1_macro   : tanpa_caption={baseline["f1_macro"]:.3f}  ->  dgn_caption={report_test["macro avg"]["f1-score"]:.3f}  '
          f'(delta {report_test["macro avg"]["f1-score"] - baseline["f1_macro"]:+.3f})')
    for name in label_names:
        key = f'f1_{name}'
        if key in baseline:
            new_val = report_test[name]['f1-score']
            print(f'  {key:16s}: tanpa_caption={baseline[key]:.3f}  ->  dgn_caption={new_val:.3f}  '
                  f'(delta {new_val - baseline[key]:+.3f})')
    print()
